# SMART100 PSA 분석 워크플로우 가이드

---

## 전체 흐름 한눈에 보기

```
시뮬레이션 결과 (Excel)
        │
        ▼  [Step 1] smart100_psa_analyzer.py
  *_results.csv  (시나리오별 RT / PRHRS / PCT / Outcome 등)
        │
        ▼  [Step 2] et_generator.py
  *_ET_result.xlsx  (Event Tree 분기 분석 결과)
        │
        ▼  [Step 3] load_to_dashboard.py
  PSA_DET_Dashboard_*_filled.xlsx  (대시보드 완성본)
```

---

## 권장 폴더 구조

```
smart100-psa-analyzer/
├── smart100_psa_analyzer.py    ← Step 1
├── et_generator.py             ← Step 2
├── load_to_dashboard.py        ← Step 3
│
├── data/
│   ├── raw/                    ← 시뮬레이션 원본 Excel (입력)
│   ├── csv/                    ← Step 1 출력 CSV
│   ├── et_result/              ← Step 2 출력 ET_result.xlsx
│   └── dashboard/              ← Step 3 출력 Dashboard.xlsx
│
├── templates/
│   └── PSA_DET_Dashboard_Template.xlsx  ← 대시보드 템플릿
│
└── mapping/
    └── smart100데이터_변수명_확인.xlsx   ← 변수 매핑 파일
```

---

## 사전 준비: 패키지 설치

처음 한 번만 실행하세요.

In [ ]:
# 필요한 패키지 설치 (처음 한 번만)
import subprocess, sys

packages = ['pandas', 'numpy', 'openpyxl', 'lxml']
for pkg in packages:
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', pkg],
                            capture_output=True, text=True)
    status = '✅' if result.returncode == 0 else '❌'
    print(f'{status} {pkg}')

---

# Step 1: 시뮬레이션 결과 분석
### `smart100_psa_analyzer.py`

**목적:** MELCOR 시뮬레이션 Excel 파일들을 읽어 각 시나리오의 주요 변수를 추출하고 CSV로 저장

**입력:** 시나리오별 시뮬레이션 결과 Excel (`.xlsx`)

**출력:** `lofw_results.csv` 또는 `sbloca_results.csv`

| 출력 컬럼 | 설명 |
|---|---|
| `Scenario` | 시나리오 파일명 |
| `Reactor_Trip` | 원자로 정지 성공 여부 (Success / Fail) |
| `RCP_Status` | RCP 상태 (Running / Coast-down / Natural Circulation) |
| `PRHRS_count` | PRHRS 작동 계통 수 (0~4) |
| `PCT_max` | 최고 피복재 온도 (K) |
| `PCT_time` | PCT 발생 시각 (s) |
| `Outcome` | 결과 판정 (OK / CD) — 기준: PCT ≥ 670K → CD |

**지원 사고 유형:**
- `1` → **LOFW** (Loss of Feedwater): RT 시점 50초 고정, feedwater 유량으로 PRHRS 판정
- `2` → **SBLOCA**: 출력 5% 이하 시점 자동 감지, HX 열출력으로 PRHRS 판정

In [ ]:
# ── Step 1 실행 예시 ─────────────────────────────────────────────
# 터미널에서 실행하는 경우:
#   python smart100_psa_analyzer.py
#   → 사고 유형 선택 (1: LOFW / 2: SBLOCA)
#   → GUI 창: 변수 매핑 파일 선택 (smart100데이터_변수명_확인.xlsx)
#   → GUI 창: 시나리오 Excel 파일들 다중 선택
#   → CSV 자동 저장

# 이 노트북에서 직접 실행하는 경우:
import sys
sys.path.insert(0, '.')   # 현재 폴더의 스크립트 import 가능하게 설정

from smart100_psa_analyzer import LOFWAnalyzer, SBLOCAAnalyzer

# LOFW 분석
analyzer = LOFWAnalyzer()
# analyzer.step1_load_mapping()    # GUI: 변수 매핑 파일 선택
# analyzer.step2_upload_files()    # GUI: 시나리오 파일 다중 선택
# analyzer.show_results()          # 결과 요약 출력
# analyzer.save_results('data/csv/LOFW_results.csv')   # CSV 저장

print('▶ 위 주석을 해제하여 순서대로 실행하세요.')

In [ ]:
# Step 1 결과 미리보기
import pandas as pd
from pathlib import Path

csv_folder = Path('data/csv')
csv_files  = list(csv_folder.glob('*.csv')) if csv_folder.exists() else []

if csv_files:
    for f in csv_files:
        df = pd.read_csv(f)
        print(f'\n📄 {f.name}  ({len(df)}행)')
        display(df.head(3))
        print(f'  OK: {(df["Outcome"]=="OK").sum()}개  |  CD: {(df["Outcome"]=="CD").sum()}개')
else:
    print('data/csv/ 폴더에 CSV 파일이 없습니다. Step 1을 먼저 실행하세요.')

---

# Step 2: Event Tree 생성
### `et_generator.py`

**목적:** Step 1에서 만든 CSV를 읽어 Event Tree 분기별로 집계하고 Excel로 저장

**입력:** `*_results.csv`

**출력:** `*_ET_result.xlsx` (시트 4개)

| 시트 | 내용 |
|---|---|
| `원본_데이터` | 정규화된 원본 데이터 전체 |
| `분기_집계` | 헤딩 조합별 시나리오 수 / 확률 / CD·OK 개수 |
| `ET_구조` | Event Tree 계층 구조 시각화 |
| `통계` | Outcome 분포 및 헤딩별 분포 요약 |

**사고유형별 헤딩 자동 결정:**

| 사고 유형 | 헤딩 |
|---|---|
| SLOCA2 | Reactor_Trip → PSIS_FEED_status → SIT_Refill_time |
| 그 외 전부 | Reactor_Trip → PRHRS_count → ADS_BLEED_count → PSIS_FEED_status → SIT_Refill_time |

**컬럼명 자동 정규화:**

| 원본 컬럼명 | 정규화 후 |
|---|---|
| `PRHRS_HX_count` | `PRHRS_count` |
| `RT_status` | `Reactor_Trip` |
| `RCP_status` | `RCP_Status` |
| `PCT_K` | `PCT_max` |
| `State` | `Outcome` |

In [ ]:
# ── Step 2 실행 예시 ─────────────────────────────────────────────
# 터미널에서 실행하는 경우 (권장):
#   python et_generator.py data/csv/ -o data/et_result/
#   python et_generator.py data/csv/LOFW_results.csv -o data/et_result/

# 이 노트북에서 직접 실행하는 경우:
from et_generator import ET_Generator
from pathlib import Path

csv_folder = Path('data/csv')
csv_files  = sorted(csv_folder.glob('*.csv')) if csv_folder.exists() else []

if csv_files:
    et_gen = ET_Generator(output_dir='data/et_result')
    et_gen.run_all(csv_files)
else:
    print('data/csv/ 폴더에 CSV 파일이 없습니다. Step 1을 먼저 실행하세요.')

In [ ]:
# Step 2 결과 확인
et_folder = Path('data/et_result')
et_files  = list(et_folder.glob('*_ET_result.xlsx')) if et_folder.exists() else []

if et_files:
    import openpyxl
    for f in et_files:
        wb = openpyxl.load_workbook(f, read_only=True)
        print(f'\n📊 {f.name}')
        print(f'   시트: {wb.sheetnames}')
        # 분기_집계 시트 미리보기
        ws = wb['분기_집계']
        rows = list(ws.iter_rows(values_only=True))
        df_branch = pd.DataFrame(rows[1:], columns=rows[0])
        display(df_branch.head(5))
        wb.close()
else:
    print('data/et_result/ 폴더에 ET_result.xlsx 파일이 없습니다. Step 2를 먼저 실행하세요.')

---

# Step 3: 대시보드 생성
### `load_to_dashboard.py`

**목적:** Step 2의 ET_result.xlsx와 대시보드 템플릿을 합쳐 완성된 PSA DET Dashboard 생성

**입력:**
- `*_ET_result.xlsx` (Step 2 출력)
- `PSA_DET_Dashboard_Template.xlsx` (템플릿 — 변경하지 않는 원본)

**출력:** `PSA_DET_Dashboard_*_filled.xlsx`

**주요 처리 내용:**

| 처리 | 내용 |
|---|---|
| Data 시트 채우기 | 원본_데이터를 템플릿 컬럼 순서에 맞게 삽입 |
| PSIS 변환 | Success→1, Fail→0, N/A→`-` |
| PCT_pass 재계산 | PCT < 1477K → Pass, 이상 → Fail |
| Calc 수식 확장 | 시나리오 수에 맞게 필터·통계 수식 자동 확장 |
| 차트 색상 패치 | 히스토그램 CD 구간(≥1477K)을 빨간색으로 표시 |
| ET_구조 시트 복사 | ET_result의 ET_구조 시트를 Dashboard에 붙여넣기 |

**CD 기준:** PCT_max ≥ **1477 K**

In [ ]:
# ── Step 3 실행 예시 ─────────────────────────────────────────────
# 터미널에서 실행하는 경우 (권장):
#   python load_to_dashboard.py -t templates/PSA_DET_Dashboard_Template.xlsx \
#                               data/et_result/ -o data/dashboard/

# 이 노트북에서 직접 실행하는 경우:
from load_to_dashboard import run_all
from pathlib import Path

template_path = Path('templates/PSA_DET_Dashboard_Template.xlsx')
et_folder     = Path('data/et_result')
et_files      = sorted(et_folder.glob('*_ET_result.xlsx')) if et_folder.exists() else []

if not template_path.exists():
    print(f'❌ 템플릿 파일을 찾을 수 없습니다: {template_path}')
elif not et_files:
    print('data/et_result/ 폴더에 ET_result.xlsx 파일이 없습니다. Step 2를 먼저 실행하세요.')
else:
    run_all(et_files, template_path, Path('data/dashboard'))

In [ ]:
# Step 3 결과 확인
dash_folder = Path('data/dashboard')
dash_files  = list(dash_folder.glob('PSA_DET_Dashboard_*_filled.xlsx')) if dash_folder.exists() else []

if dash_files:
    import openpyxl
    for f in dash_files:
        wb = openpyxl.load_workbook(f, read_only=True)
        print(f'\n📊 {f.name}')
        print(f'   시트: {wb.sheetnames}')
        ws = wb['Data']
        rows = list(ws.iter_rows(min_row=1, max_row=4, values_only=True))
        print(f'   헤더: {rows[0]}')
        print(f'   샘플: {rows[1]}')
        wb.close()
else:
    print('data/dashboard/ 폴더에 Dashboard 파일이 없습니다. Step 3을 먼저 실행하세요.')

---

# 전체 자동 실행 (원스텝)

Step 1 완료(CSV 준비) 후, Step 2~3을 한 번에 실행합니다.

In [ ]:
from et_generator import ET_Generator
from load_to_dashboard import run_all
from pathlib import Path

# ── 경로 설정 ──────────────────────────────────────────────────────
CSV_FOLDER      = Path('data/csv')           # Step 1 출력 CSV 폴더
ET_FOLDER       = Path('data/et_result')     # Step 2 출력 폴더
DASHBOARD_FOLDER= Path('data/dashboard')     # Step 3 출력 폴더
TEMPLATE_PATH   = Path('templates/PSA_DET_Dashboard_Template.xlsx')

# ── Step 2: ET 생성 ───────────────────────────────────────────────
csv_files = sorted(CSV_FOLDER.glob('*.csv'))
if not csv_files:
    raise FileNotFoundError(f'{CSV_FOLDER} 에 CSV 파일이 없습니다.')

print('[ Step 2: ET 생성 ]')
et_gen = ET_Generator(output_dir=ET_FOLDER)
et_gen.run_all(csv_files)

# ── Step 3: 대시보드 생성 ─────────────────────────────────────────
et_files = sorted(ET_FOLDER.glob('*_ET_result.xlsx'))
if not et_files:
    raise FileNotFoundError(f'{ET_FOLDER} 에 ET_result 파일이 없습니다.')
if not TEMPLATE_PATH.exists():
    raise FileNotFoundError(f'템플릿 파일을 찾을 수 없습니다: {TEMPLATE_PATH}')

print('\n[ Step 3: 대시보드 생성 ]')
run_all(et_files, TEMPLATE_PATH, DASHBOARD_FOLDER)

print('\n✅ 모든 단계 완료!')
print(f'   결과 위치: {DASHBOARD_FOLDER.resolve()}')

---

# 터미널 명령어 모음 (빠른 참조)

## 기본 실행 (현재 폴더에서 바로 사용)

```bash
# ── Step 1 ──────────────────────────────────────────────────────
python smart100_psa_analyzer.py
# → 사고 유형 선택 → GUI 파일 선택 → CSV 자동 저장

# ── Step 2 ──────────────────────────────────────────────────────
# 현재 폴더의 모든 CSV → 현재 폴더에 ET_result.xlsx 저장
python et_generator.py

# ── Step 3 ──────────────────────────────────────────────────────
# templates 폴더에 템플릿 파일을 넣어두면 이 한 줄로 실행 가능
python load_to_dashboard.py -t templates\PSA_DET_Dashboard_template.xlsx
```

> **템플릿 파일 위치:** `smart100-psa-analyzer/templates/PSA_DET_Dashboard_template.xlsx`

---

## 폴더 구분해서 정리할 때

```bash
# ── Step 1 ──────────────────────────────────────────────────────
python smart100_psa_analyzer.py

# ── Step 2 ──────────────────────────────────────────────────────
# 특정 폴더의 CSV → 출력 폴더 지정
python et_generator.py data/csv/ -o data/et_result/

# 특정 파일만
python et_generator.py data/csv/LOFW_results.csv -o data/et_result/

# ── Step 3 ──────────────────────────────────────────────────────
# 폴더 전체 + 출력 폴더 지정
python load_to_dashboard.py -t templates\PSA_DET_Dashboard_template.xlsx data/et_result/ -o data/dashboard/

# 특정 파일만
python load_to_dashboard.py -t templates\PSA_DET_Dashboard_template.xlsx data/et_result/LOFW_ET_result.xlsx -o data/dashboard/
```

---

## 자주 발생하는 오류

| 오류 메시지 | 원인 | 해결 방법 |
|---|---|---|
| `CSV 인코딩을 감지할 수 없습니다` | 특수 인코딩 | 파일을 UTF-8로 다시 저장 |
| `헤딩 'X' 없음 → 건너뜀` | CSV에 해당 컬럼 없음 | Step 1 결과 컬럼명 확인 |
| `원본_데이터 시트가 없습니다` | ET_result.xlsx가 아닌 파일 선택 | Step 2 출력 파일인지 확인 |
| `the following arguments are required: -t` | 템플릿 경로 미지정 | `-t templates\PSA_DET_Dashboard_template.xlsx` 추가 |
| `템플릿 파일을 찾을 수 없습니다` | 경로 오류 | templates 폴더에 파일이 있는지 확인 |